# 16. Conditioning, DiT-B/2, and SD3-Medium MMDiT

Only tensor widths and image/token counts are reduced. DiT-B/2 keeps 12 blocks/12 heads,
adaLN-Zero, and fixed 2D sin-cos patch positions. SD3-Medium keeps 24 joint blocks/24 heads,
separate image/context parameters, pooled-text + timestep conditioning, the final context-pre-only
block, final adaptive normalization, output projection, and unpatchify.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(7)
device = torch.device('cpu')


In [ ]:
def sincos_1d(dim, positions):
    assert dim % 2 == 0
    omega = torch.arange(dim // 2, device=positions.device, dtype=torch.float32)
    omega = 1.0 / 10000 ** (omega / (dim / 2))
    angles = positions.reshape(-1, 1).float() * omega[None]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)

def sincos_2d(dim, grid_size, device):
    assert dim % 4 == 0
    y, x = torch.meshgrid(
        torch.arange(grid_size, device=device),
        torch.arange(grid_size, device=device),
        indexing='ij',
    )
    parts = [
        sincos_1d(dim // 2, x.reshape(-1)),
        sincos_1d(dim // 2, y.reshape(-1)),
    ]
    return torch.cat(parts, dim=-1)[None]

def timestep_embedding(timestep, dim):
    half = dim // 2
    frequency = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=timestep.device, dtype=torch.float32)
        / half
    )
    angle = timestep.float()[:, None] * frequency[None]
    return torch.cat([angle.cos(), angle.sin()], dim=-1)

def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


## 1. DiT-B/2


In [ ]:
class DiTBlock(nn.Module):

    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.heads = heads
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * dim, dim),
        )
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.ada[-1].weight)
        nn.init.zeros_(self.ada[-1].bias)

    def forward(self, hidden, condition):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.ada(condition).chunk(6, dim=-1)
        attention_input = modulate(self.norm1(hidden), shift_a, scale_a)
        attention_output, _ = self.attention(
            attention_input, attention_input, attention_input, need_weights=False
        )
        hidden = hidden + gate_a[:, None] * attention_output
        mlp_input = modulate(self.norm2(hidden), shift_m, scale_m)
        return hidden + gate_m[:, None] * self.mlp(mlp_input)

class SmallWidthDiTB2(nn.Module):

    def __init__(self, image_size=8, channels=4, dim=48, classes=10):
        super().__init__()
        self.image_size = image_size
        self.patch_size = 2
        self.patch = nn.Conv2d(channels, dim, 2, stride=2)
        position = sincos_2d(dim, image_size // 2, torch.device('cpu'))
        self.register_buffer('position', position, persistent=False)
        self.time = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.label = nn.Embedding(classes + 1, dim)
        self.blocks = nn.ModuleList([DiTBlock(dim, 12) for _ in range(12)])
        self.final_norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
        self.final_ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.output = nn.Linear(dim, 2 * 2 * channels * 2)

    def forward(self, image, timestep, label):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position.to(hidden.device, hidden.dtype)
        condition = self.time(timestep_embedding(timestep, hidden.size(-1))) + self.label(label)
        for block in self.blocks:
            hidden = block(hidden, condition)
        shift, scale = self.final_ada(condition).chunk(2, dim=-1)
        patches = self.output(modulate(self.final_norm(hidden), shift, scale)).transpose(1, 2)
        return F.fold(patches, (self.image_size, self.image_size), kernel_size=2, stride=2)
dit = SmallWidthDiTB2()
assert len(dit.blocks) == 12
assert all((block.heads == 12 for block in dit.blocks))
assert not isinstance(dit.position, nn.Parameter)


## 2. SD3-Medium MMDiT with released conditioning/output path


In [ ]:
class AdaLNZero(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))

    def forward(self, hidden, condition):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = (
            self.modulation(condition).chunk(6, dim=-1)
        )
        return (modulate(self.norm(hidden), shift_a, scale_a), gate_a, shift_m, scale_m, gate_m)

class AdaLNContinuous(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))

    def forward(self, hidden, condition):
        shift, scale = self.modulation(condition).chunk(2, dim=-1)
        return modulate(self.norm(hidden), shift, scale)

class SD3Stream(nn.Module):

    def __init__(self, dim=48, heads=24, pre_only=False):
        super().__init__()
        self.pre_only = pre_only
        self.heads = heads
        self.head_dim = dim // heads
        self.norm1 = AdaLNContinuous(dim) if pre_only else AdaLNZero(dim)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        if not pre_only:
            self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-06)
            self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * dim, dim),
        )

    def project(self, hidden):
        batch, length, dim = hidden.shape
        qkv = self.qkv(hidden).view(batch, length, 3, self.heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)

class JointBlock(nn.Module):

    def __init__(self, dim=48, heads=24, context_pre_only=False):
        super().__init__()
        self.context_pre_only = context_pre_only
        self.image = SD3Stream(dim, heads, pre_only=False)
        self.context = SD3Stream(dim, heads, pre_only=context_pre_only)

    @staticmethod
    def merge(heads):
        return heads.transpose(1, 2).contiguous().flatten(2)

    def forward(self, image, context, condition):
        image_n, image_gate, image_shift, image_scale, image_mlp_gate = (
            self.image.norm1(image, condition)
        )
        if self.context_pre_only:
            context_n = self.context.norm1(context, condition)
        else:
            context_values = self.context.norm1(context, condition)
            context_n, context_gate, context_shift, context_scale, context_mlp_gate = (
                context_values
            )
        iq, ik, iv = self.image.project(image_n)
        cq, ck, cv = self.context.project(context_n)
        query = torch.cat([iq, cq], dim=2)
        key = torch.cat([ik, ck], dim=2)
        value = torch.cat([iv, cv], dim=2)
        joint = F.scaled_dot_product_attention(query, key, value)
        image_length = image.size(1)
        image_attention = self.merge(joint[:, :, :image_length])
        context_attention = self.merge(joint[:, :, image_length:])
        image = image + image_gate[:, None] * self.image.output(image_attention)
        image_mlp_input = modulate(self.image.norm2(image), image_shift, image_scale)
        image = image + image_mlp_gate[:, None] * self.image.mlp(image_mlp_input)
        if not self.context_pre_only:
            context = context + context_gate[:, None] * self.context.output(context_attention)
            context_mlp_input = modulate(self.context.norm2(context), context_shift, context_scale)
            context = context + context_mlp_gate[:, None] * self.context.mlp(context_mlp_input)
        return (image, context)


In [ ]:
class SmallWidthSD3Medium(nn.Module):

    def __init__(
        self, sample_size=8, in_channels=16, out_channels=16,
        dim=48, context_dim=32, pooled_dim=24,
    ):
        super().__init__()
        self.sample_size = sample_size
        self.patch_size = 2
        self.out_channels = out_channels
        self.patch = nn.Conv2d(in_channels, dim, self.patch_size, stride=self.patch_size)
        position = sincos_2d(dim, sample_size // self.patch_size, torch.device('cpu'))
        self.register_buffer('position', position, persistent=False)
        self.context_projection = nn.Linear(context_dim, dim)
        self.time_projection = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.pooled_projection = nn.Sequential(
            nn.Linear(pooled_dim, dim), nn.SiLU(), nn.Linear(dim, dim)
        )
        self.blocks = nn.ModuleList([
            JointBlock(dim, 24, context_pre_only=index == 23)
            for index in range(24)
        ])
        self.norm_out = AdaLNContinuous(dim)
        self.proj_out = nn.Linear(dim, self.patch_size * self.patch_size * out_channels)

    def forward(self, image, context, pooled_text, timestep):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position.to(hidden.device, hidden.dtype)
        context = self.context_projection(context)
        time_condition = self.time_projection(timestep_embedding(timestep, hidden.size(-1)))
        pooled_condition = self.pooled_projection(pooled_text)
        condition = time_condition + pooled_condition
        for block in self.blocks:
            hidden, context = block(hidden, context, condition)
        hidden = self.norm_out(hidden, condition)
        patches = self.proj_out(hidden)
        batch = patches.size(0)
        grid = self.sample_size // self.patch_size
        patches = patches.view(
            batch, grid, grid, self.patch_size, self.patch_size, self.out_channels
        )
        image = torch.einsum('bhwpqc->bchpwq', patches)
        image = image.reshape(
            batch, self.out_channels, self.sample_size, self.sample_size
        )
        return image
sd3 = SmallWidthSD3Medium()
assert len(sd3.blocks) == 24
assert all((block.image.heads == 24 for block in sd3.blocks))
assert sd3.blocks[-1].context_pre_only
assert hasattr(sd3, 'pooled_projection')
assert hasattr(sd3, 'norm_out')
assert hasattr(sd3, 'proj_out')
assert not isinstance(sd3.position, nn.Parameter)
image = torch.randn(1, 16, 8, 8)
context = torch.randn(1, 5, 32)
pooled = torch.randn(1, 24)
timestep = torch.tensor([500])
output = sd3(image, context, pooled, timestep)
output.square().mean().backward()
assert output.shape == (1, 16, 8, 8)


## Audit result

SD3 no longer stops at MMDiT tokens. It now includes pooled-text + timestep conditioning,
the final context-pre-only block, final adaptive normalization, patch output projection, and
unpatchify back to the latent image.
